# Line Crossing Count: Vehicles

This notebook counts vehicles crossing a line on a highway, split by direction and by vehicle type. It builds on the detection and tracking notebook:

1. Detect vehicles with [RF-DETR](https://github.com/roboflow/rf-detr).
2. Track them with ByteTrack from [trackers](https://github.com/roboflow/trackers) so each vehicle has a stable ID.
3. Count each ID once when it crosses a virtual line, using Supervision's `sv.LineZone`.

Tracking is what makes counting possible: without IDs we couldn't tell whether the car on the line in frame 10 is the same car as in frame 11.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import supervision as sv
from rfdetr import RFDETRNano
from rfdetr.assets.coco_classes import COCO_CLASSES
from supervision.assets import VideoAssets, download_assets
from trackers import ByteTrackTracker

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## Download the video

`VEHICLES` is a 4K clip of a highway. Traffic on the left carriageway drives towards the camera; traffic on the right drives away from it.

In [ ]:
SOURCE_VIDEO_PATH = download_assets(VideoAssets.VEHICLES, directory=DATA_DIR)
TARGET_VIDEO_PATH = OUTPUT_DIR / "vehicles_line_count.mp4"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
video_info

## Place the line

A `LineZone` is defined by two points in pixel coordinates. We draw a horizontal line across both carriageways, low in the frame where vehicles are large and easy to detect.

Annotation sizes are scaled to the resolution with `sv.calculate_optimal_text_scale` and `sv.calculate_optimal_line_thickness`; the defaults are tuned for ~720p and would be unreadable on a 4K frame.

In [ ]:
LINE_START = sv.Point(0, 1500)
LINE_END = sv.Point(video_info.width, 1500)

text_scale = sv.calculate_optimal_text_scale(video_info.resolution_wh)
thickness = sv.calculate_optimal_line_thickness(video_info.resolution_wh)

frame = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))
preview = sv.draw_line(frame.copy(), LINE_START, LINE_END, color=sv.Color.RED, thickness=thickness * 2)
sv.plot_image(preview, size=(12, 7))

## Detect vehicles

We keep the COCO vehicle classes: car, motorcycle, bus and truck.

In [ ]:
model = RFDETRNano()

VEHICLE_CLASSES = ["car", "motorcycle", "bus", "truck"]
VEHICLE_CLASS_IDS = [k for k, v in COCO_CLASSES.items() if v in VEHICLE_CLASSES]


def detect(frame, threshold):
    detections = model.predict(frame[:, :, ::-1].copy(), threshold=threshold)
    return detections[np.isin(detections.class_id, VEHICLE_CLASS_IDS)]

In [ ]:
detections = detect(frame, threshold=0.5)

box_annotator = sv.BoxAnnotator(thickness=thickness)
label_annotator = sv.LabelAnnotator(text_scale=text_scale, text_thickness=thickness, smart_position=True)

labels = [
    f"{COCO_CLASSES[class_id]} {confidence:.2f}"
    for class_id, confidence in zip(detections.class_id, detections.confidence)
]

annotated_frame = box_annotator.annotate(frame.copy(), detections)
annotated_frame = label_annotator.annotate(annotated_frame, detections, labels)
sv.plot_image(annotated_frame, size=(12, 7))

## Count line crossings

`LineZone.trigger` compares each tracked object's position with the previous frame and reports which IDs crossed the line and in which direction. Each ID is counted only once per direction.

- **Triggering anchor.** By default an object only counts once all four corners of its box have crossed. We use the bottom-centre point instead (roughly where the wheels touch the road), which is more intuitive with this camera angle.
- **Direction.** "In" and "out" depend on the order of the two points. With the line drawn left to right, "in" is moving up the frame (away from the camera) and "out" is moving down (towards it). `custom_in_text` / `custom_out_text` rename them on the annotation.

In [ ]:
tracker = ByteTrackTracker(frame_rate=video_info.fps)
line_zone = sv.LineZone(start=LINE_START, end=LINE_END, triggering_anchors=[sv.Position.BOTTOM_CENTER])

box_annotator = sv.BoxAnnotator(thickness=thickness, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(
    text_scale=text_scale, text_thickness=thickness, color_lookup=sv.ColorLookup.TRACK, smart_position=True
)
trace_annotator = sv.TraceAnnotator(
    thickness=thickness, color_lookup=sv.ColorLookup.TRACK, trace_length=video_info.fps * 2, position=sv.Position.BOTTOM_CENTER
)
line_zone_annotator = sv.LineZoneAnnotator(
    thickness=thickness,
    text_thickness=thickness,
    text_scale=text_scale,
    custom_in_text="away",
    custom_out_text="towards",
)


def callback(frame, index):
    detections = detect(frame, threshold=0.3)
    detections = tracker.update(detections)
    detections = detections[detections.tracker_id != -1]
    line_zone.trigger(detections)

    labels = [
        f"#{tracker_id} {COCO_CLASSES[class_id]}"
        for tracker_id, class_id in zip(detections.tracker_id, detections.class_id)
    ]

    annotated = trace_annotator.annotate(frame.copy(), detections)
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    annotated = line_zone_annotator.annotate(annotated, line_counter=line_zone)
    return annotated

In [ ]:
sv.process_video(
    source_path=str(SOURCE_VIDEO_PATH),
    target_path=str(TARGET_VIDEO_PATH),
    callback=callback,
    show_progress=True,
)

## Results

`LineZone` keeps the totals as well as a per-class breakdown.

In [ ]:
print(f"Towards the camera: {line_zone.out_count}")
print(f"Away from the camera: {line_zone.in_count}")

counts = pd.DataFrame(
    {
        "towards": {COCO_CLASSES[k]: v for k, v in line_zone.out_count_per_class.items()},
        "away": {COCO_CLASSES[k]: v for k, v in line_zone.in_count_per_class.items()},
    }
).fillna(0).astype(int)
counts.loc["total"] = counts.sum()
counts

In [ ]:
frames = list(sv.get_video_frames_generator(TARGET_VIDEO_PATH, stride=video_info.total_frames // 5))[1:5]
sv.plot_images_grid(frames, grid_size=(2, 2), size=(16, 9))

Open `output/vehicles_line_count.mp4` to watch the full video with the running counts.

**Things to try**

- Move the line higher up the frame. Vehicles get smaller and detections less reliable; how do the counts change?
- Use two `LineZone`s, one per carriageway, instead of relying on direction.
- Try the `VEHICLES_2` asset, or `sv.LineZoneAnnotatorMulticlass` to draw the per-class counts on the video.